# Generate Audio File from example sentences

In [ ]:
import os
import subprocess
import wave

from dotenv import load_dotenv
from google import genai
from google.genai import types
from gtts import gTTS

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)


def generate_audio_for_word(content: str, language: str = "en"):
    """Generate audio for a given word using Gemini TTS"""
    # Passing the text and language to the engine,
    # here we have marked slow=False. Which tells
    # the module that the converted audio should
    # have a high speed
    myobj = gTTS(text=content, lang=language, slow=False)
    return myobj


def concatenate_audio_files(mp3_files: list[str], output_file: str) -> None:
    """Concatenate multiple audio files into one"""
    # ffmpeg用に一時的なテキストファイルを作成
    with open("file_list.txt", "w", encoding="utf-8") as f:
        for mp3 in mp3_files:
            f.write(f"file '{os.path.abspath(mp3)}'\n")

    # ffmpegコマンドで結合（出力はcombined.mp3）
    subprocess.run(
        [
            "/opt/homebrew/bin/ffmpeg",
            "-f",
            "concat",
            "-safe",
            "0",
            "-i",
            "file_list.txt",
            "-c",
            "copy",
            output_file,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    # 一時ファイルを削除
    os.remove("file_list.txt")
    for mp3 in mp3_files:
        os.remove(mp3)

In [ ]:
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import GoogleGenerativeAI

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

# Gemini APIの初期化
llm = GoogleGenerativeAI(
    model="gemini-2.0-flash", google_api_key=GOOGLE_API_KEY, rate_limiter=rate_limiter
)

# 例文生成用のプロンプトテンプレート
example_prompt = PromptTemplate.from_template(
    """Generate the simple one sentence discription of the English word "{word}" in Japanese.
        
    # example for `comprise`:
    含む、構成する

    # example for `accrue`:
    累積する、増える、生じる
    """
)

# Chain
chain = example_prompt | llm | StrOutputParser()


def generate_short_explanation(word: str) -> str:
    """単語に対して例文を生成する"""
    output = chain.invoke({"word": word})
    return output


def generate_content_from_db(word: str, examples: list[str]) -> str:
    """Generate content from the database for a given word and examples"""
    japanese_explanation: str = generate_short_explanation(word)
    content = f"""TTS the following sentence. Please read English word and examples as an native English speaker, while Japanese explanation should be read as a native Japanese speaker.:
    # sentence
    {word} {japanese_explanation}
    {examples[0]}
    {examples[1]}
    {examples[2]}
    """
    return content

In [7]:
import time

import pandas as pd

# 10960
# 10792

df_word = pd.read_csv("../word_data/lv12.csv")
df_example = pd.read_csv("../generate_examples/lv12_with_examples.csv")

for index, row in df_word.iterrows():
    # if index < 942:
    #     continue
    if index != 604:
        continue
    print(f"Processing word {index + 1}/{len(df_word)}: {row['word']}")
    time.sleep(10)
    examples = df_example[df_example["word_id"] == row["word_id"]]["example"].tolist()
    word: str = str(row["word"])
    # word
    word_audio = generate_audio_for_word(word, language="en")
    #
    ja_audio = generate_audio_for_word(generate_short_explanation(word), language="ja")
    # examples
    examples = f"""
    {examples[0]}
    {examples[1]}
    {examples[2]}
    """
    example_audios = generate_audio_for_word(examples, language="en")
    word_audio.save("word.mp3")
    ja_audio.save("ja.mp3")
    example_audios.save("examples.mp3")
    concatenate_audio_files(
        ["word.mp3", "ja.mp3", "examples.mp3"], f"{word}_{row['word_id']}.mp3"
    )

Processing word 605/1000: stupor


In [ ]:
for index, row in df_word.iterrows():
    word = row["word"]
    word_id = row["word_id"]
    if not os.path.exists(f"./lv12/{word}_{word_id}.mp3"):
        print(f"Skipping {word}@{word_id} as audio file does not exist.")

Skipping stupor@11605 as audio file does not exist.
